# Load test analysis

Reads `metrics.csv` from the repo root (generated by `perf/load_test.py`). Compare TTFT, end-to-end latency, and throughput across concurrency, prompt length (short vs long), optional repeated prompts (`cache_on`), and stop-sequence settings.

In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

csv_path = Path("../metrics.csv")
df = pd.read_csv(csv_path)
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for profile in df["profile"].unique():
    sub = df[df["profile"] == profile]
    ax.scatter(sub["concurrency"], sub["latency_ms_p50"], label=profile)
ax.set_xlabel("Concurrency")
ax.set_ylabel("p50 latency (ms)")
ax.legend()
ax.set_title("Latency vs concurrency (by prompt profile)")
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(df)), df["ttft_ms_p50"])
ax.set_xticks(range(len(df)))
ax.set_xticklabels(df["run_id"], rotation=45, ha="right")
ax.set_ylabel("TTFT p50 (ms)")
ax.set_title("Time-to-first-token across experiment rows")
plt.tight_layout()

In [ ]:
# Throughput vs client batch size (same as concurrency in load_test.py)
fig, ax = plt.subplots(figsize=(8, 4))
for profile in df["profile"].unique():
    sub = df[df["profile"] == profile]
    ax.scatter(sub["client_batch_size"], sub["throughput_tokens_per_sec"], label=profile)
ax.set_xlabel("client_batch_size / concurrency")
ax.set_ylabel("throughput (tokens/s)")
ax.legend()
ax.set_title("Throughput vs batch size")
plt.tight_layout()

## Commentary

- **Higher concurrency** usually increases queueing; TTFT may grow if the server is saturated.
- **`cache_on`**: repeating the same prompt exercises prefix caching (if enabled in vLLM) and removes prompt variability so latency reflects decode/cache behavior.
- **Stop sequences** (`double_newline` vs none) shorten generations, improving throughput and lowering total latency when answers are verbose.
- **Short vs long** profiles stress prefill vs decode; long prompts raise prefill time and can dominate TTFT.